# 5. TabPFN
(version 2.5). Needs a GPU

```
pip install tabpfn
```

In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, matthews_corrcoef, accuracy_score
from sklearn.inspection import permutation_importance
import warnings
warnings.filterwarnings("ignore", message="X does not have valid feature names")

### round_trip: read the numbers back exactly as notebook 1 saved them ###
data = pd.read_csv("model_data.csv", parse_dates=["Date"], float_precision="round_trip")

FEATURES_A = ["ret_lag1", "ret_lag2", "ret_lag3", "ret_lag4", "ret_lag5",
              "hl_range", "intraday_ret", "dlog_volume"]
FEATURES_B = FEATURES_A + ["dist_sma10", "dist_sma50", "rsi14", "macd_rel", "macd_signal_rel"]
FEATURE_SETS = {"A": FEATURES_A, "B": FEATURES_B}

TEST_YEARS = [2020, 2021, 2022, 2023, 2024]

import time
import torch
from tabpfn import TabPFNClassifier
from tabpfn.constants import ModelVersion

### settings ###
SEEDS = [0]     # one seed because of the computing time; use [0, 1, 2, 3, 4] for five seeds
CHUNK = 1000    # test rows predicted at once; a larger chunk (e.g. 3000) is faster if memory allows

print("Apple GPU available:", torch.backends.mps.is_available())

Apple GPU available: True


In [2]:
### the three evaluation settings: (setting, held-out unit, training stocks, test stocks) ###

TECH = ["AAPL", "MSFT", "AMZN", "GOOGL", "TSLA"]
ENERGY = ["XOM", "CVX", "COP"]
FOOD = ["PEP", "KO", "MDLZ"]
TECH_CROSS_SECTOR = ["AAPL", "MSFT", "GOOGL"]   # three tech stocks, so every sector has three

units = [("within", "all", TECH + ENERGY + FOOD, TECH + ENERGY + FOOD)]

for stock in TECH:
    units.append(("cross_stock", stock, [s for s in TECH if s != stock], [stock]))

units.append(("cross_sector", "Technology", ENERGY + FOOD, TECH_CROSS_SECTOR))
units.append(("cross_sector", "Energy", TECH_CROSS_SECTOR + FOOD, ENERGY))
units.append(("cross_sector", "Food & Beverage", TECH_CROSS_SECTOR + ENERGY, FOOD))

pd.DataFrame(units, columns=["setting", "held_out", "train_stocks", "test_stocks"])

,setting,held_out,train_stocks,test_stocks
0,within,all,"[AAPL, MSFT, AMZN, GOOGL, TSLA, XOM, CVX, COP,...","[AAPL, MSFT, AMZN, GOOGL, TSLA, XOM, CVX, COP,..."
1,cross_stock,AAPL,"[MSFT, AMZN, GOOGL, TSLA]",[AAPL]
2,cross_stock,MSFT,"[AAPL, AMZN, GOOGL, TSLA]",[MSFT]
3,cross_stock,AMZN,"[AAPL, MSFT, GOOGL, TSLA]",[AMZN]
4,cross_stock,GOOGL,"[AAPL, MSFT, AMZN, TSLA]",[GOOGL]
5,cross_stock,TSLA,"[AAPL, MSFT, AMZN, GOOGL]",[TSLA]
6,cross_sector,Technology,"[XOM, CVX, COP, PEP, KO, MDLZ]","[AAPL, MSFT, GOOGL]"
7,cross_sector,Energy,"[AAPL, MSFT, GOOGL, PEP, KO, MDLZ]","[XOM, CVX, COP]"
8,cross_sector,Food & Beverage,"[AAPL, MSFT, GOOGL, XOM, CVX, COP]","[PEP, KO, MDLZ]"


In [3]:
### train/test split with a one-day embargo ###

def remove_last_day(df):
    # the target of the last training day uses the first price of the next year
    return df[df["Date"] != df["Date"].max()]

def split(test_year, train_stocks, test_stocks):
    train = data[(data["year"] < test_year) & data["Ticker"].isin(train_stocks)]
    train = remove_last_day(train)
    test = data[(data["year"] == test_year) & data["Ticker"].isin(test_stocks)]
    return train, test

def metrics(y, proba):
    pred = (proba >= 0.5).astype(int)
    return {"auc": roc_auc_score(y, proba) if len(np.unique(proba)) > 1 else 0.5,
            "mcc": matthews_corrcoef(y, pred),
            "acc": accuracy_score(y, pred),
            "up_rate": pred.mean(),
            "recall_up": pred[y == 1].mean(),
            "recall_down": 1 - pred[y == 0].mean()}

In [4]:
### test run: one evaluation, to measure the time ###
### the full run takes about 47 x this time with five seeds, about 9 x with one seed ###

train, test = split(2024, TECH + ENERGY + FOOD, TECH + ENERGY + FOOD)
start = time.time()
model = TabPFNClassifier.create_default_for_version(ModelVersion.V2_5, random_state=0)
model.fit(train[FEATURES_A], train["Target"])
proba = np.concatenate([model.predict_proba(test[FEATURES_A].iloc[i:i + CHUNK])[:, 1]
                        for i in range(0, len(test), CHUNK)])
print(f"{time.time() - start:.0f} seconds, ROC-AUC {roc_auc_score(test['Target'], proba):.3f}")

tabpfn-v2.5-classifier-v2.5_default.ckpt: reconstructing file:   0%|          |  0.00B / 42.9MB            

tabpfn-v2.5-classifier-v2.5_default.ckpt: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

283 seconds, ROC-AUC 0.472


In [5]:
### train and test (no tuning) ###

results, predictions, importance = [], [], []

for test_year in TEST_YEARS:
    for setting, held_out, train_stocks, test_stocks in units:
        train, test = split(test_year, train_stocks, test_stocks)
        info = {"test_year": test_year, "setting": setting, "held_out": held_out,
                "n_train": len(train), "n_test": len(test)}

        for feature_set, features in FEATURE_SETS.items():
            probas = []
            for seed in SEEDS:
                model = TabPFNClassifier.create_default_for_version(ModelVersion.V2_5, random_state=seed)
                model.fit(train[features], train["Target"])
                proba = np.concatenate([model.predict_proba(test[features].iloc[i:i + CHUNK])[:, 1]
                                        for i in range(0, len(test), CHUNK)])
                probas.append(proba)
                results.append({**info, "model": "tabpfn", "features": feature_set, "seed": seed,
                                **metrics(test["Target"].to_numpy(), proba)})

            predictions.append(test[["Date", "Ticker", "Target", "next_ret"]]
                               .assign(**info, model="tabpfn", features=feature_set,
                                       proba=np.mean(probas, axis=0)))

    print(test_year, "done")

2020 done


2021 done


2022 done


2023 done


2024 done


In [6]:
### save ###

results = pd.DataFrame(results)
predictions = pd.concat(predictions, ignore_index=True)

results.to_csv("results/results_tabpfn.csv", index=False)
predictions.to_csv("results/predictions_tabpfn.csv.gz", index=False)

results.groupby(["setting", "features"])[["auc", "mcc", "acc", "up_rate"]].mean().round(3)

auc    mcc    acc  up_rate
setting      features                              
cross_sector A         0.515  0.025  0.523    0.748
             B         0.512  0.029  0.522    0.731
cross_stock  A         0.502  0.011  0.527    0.900
             B         0.501 -0.009  0.523    0.894
within       A         0.505  0.007  0.520    0.838
             B         0.504  0.003  0.518    0.826